In [22]:
import numpy as np
import evaluate

from datasets import load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification,TrainingArguments,Trainer,DataCollatorWithPadding




In [4]:
dataset = load_dataset("fancyzhx/yelp_polarity")

dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 560000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 38000
    })
})

Tokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")



In [6]:

def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128
    )



In [7]:
tokenized_dataset = dataset.map(tokenize_function,batched=True)



Map:   0%|          | 0/560000 [00:00<?, ? examples/s]

Map:   0%|          | 0/38000 [00:00<?, ? examples/s]

In [8]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(6000))

test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(2000))



In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)



Model

In [10]:
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased",num_labels=2)



Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Accuracy Metric

In [11]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )



In [12]:
training_args = TrainingArguments(
    output_dir="./yelp_results",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_steps=100,

    load_best_model_at_end=True,

    fp16=True
)



In [14]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy
1,0.285668,0.235041,0.907000
2,0.185278,0.258706,0.911500
3,0.106628,0.327343,0.912000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1125, training_loss=0.20104918755425347, metrics={'train_runtime': 88.77, 'train_samples_per_second': 202.771, 'train_steps_per_second': 12.673, 'total_flos': 596103293952000.0, 'train_loss': 0.20104918755425347, 'epoch': 3.0})

Evaluate Model

In [15]:
results = trainer.evaluate()

results



Training Loss,Validation Loss,Epoch,Accuracy
0.106628,0.235041,3,0.907000


{'eval_loss': 0.23504050076007843, 'eval_accuracy': 0.907}

In [18]:
with open('evaluation_results.txt', 'w') as f:
    for key, value in results.items():
        f.write(f"{key}: {value}\n")


In [16]:
model.save_pretrained("./fine_tuned_yelp_model")

tokenizer.save_pretrained("./fine_tuned_yelp_model")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_yelp_model/tokenizer_config.json',
 './fine_tuned_yelp_model/tokenizer.json')

test

In [20]:


from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="./fine_tuned_yelp_model",
    tokenizer="./fine_tuned_yelp_model"
)

sample_reviews = [

    "The food was amazing and service was excellent.",

    "Worst restaurant experience ever.",

    "Very clean place and friendly staff.",

    "I will never visit this hotel again."
]

predictions = classifier(sample_reviews)

labels = {0: "Negative Review", 1: "Positive Review"}

print("\nPredictions:\n")

for text, pred in zip(sample_reviews, predictions):

    predicted_label = int(pred["label"].split("_")[-1])

    print(f"Review: {text}")

    print(f"Prediction: {labels[predicted_label]}")

    print(f"Confidence Score: {round(pred['score'], 4)}")

    print("-" * 60)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


Predictions:

Review: The food was amazing and service was excellent.
Prediction: Positive Review
Confidence Score: 0.9857
------------------------------------------------------------
Review: Worst restaurant experience ever.
Prediction: Negative Review
Confidence Score: 0.9743
------------------------------------------------------------
Review: Very clean place and friendly staff.
Prediction: Positive Review
Confidence Score: 0.9763
------------------------------------------------------------
Review: I will never visit this hotel again.
Prediction: Negative Review
Confidence Score: 0.9347
------------------------------------------------------------
